# AI Clinical Decision Support Lite – Day 1 Ingestion Pipeline
### Google Colab Notebook: PDF Parsing, Text Cleaning, 800/100 Section-Aware Chunking, and Vector Retrieval

**Event**: Orange Digital Center x CREATIVA AI Hackathon  
**Goal**: Build an evidence-grounded Retrieval-Augmented Generation (RAG) system for official medical guidelines (*NICE Asthma* & *WHO Asthma*).

> **Core Philosophy**: *Fluent Answer ≠ Safe Answer*. Every clinical recommendation must trace back to official, page-and-section-verified guidelines with zero hallucinations.

## 1. Environment & Library Setup
Install required PDF parsing and tokenizing dependencies.

In [1]:
%pip install pypdf tiktoken chromadb sentence-transformers

import os
import re
import json
import sys
import numpy as np
from typing import List, Dict, Any, Tuple

print("Libraries successfully imported!")

  Using cached chromadb-1.5.9-cp39-abi3-win_amd64.whl.metadata (5.1 kB)
  Using cached sentence_transformers-5.7.0-py3-none-any.whl.metadata (18 kB)
  Using cached pydantic-2.13.4-py3-none-any.whl.metadata (109 kB)
  Using cached aiosignal-1.4.0-py3-none-any.whl.metadata (3.7 kB)
  Using cached frozenlist-1.8.0-cp312-cp312-win_amd64.whl.metadata (21 kB)
  Using cached multidict-6.7.1-cp312-cp312-win_amd64.whl.metadata (5.5 kB)
  Using cached annotated_types-0.8.0-py3-none-any.whl.metadata (15 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.6.1-py3-none-any.whl.metadata (6.8 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
   ---------------------------------------- 0.0/23.5 MB ? eta -:--:--
   - -------------------------------------- 0.8/23.5 MB 3.7 MB/s eta 0:00:07
   -- ------------------------------------- 1.3/23.5 MB 3.5 MB/s eta 0:00:07
   --- ------------------------------------ 2.1/23.5 MB 3.8 MB/s eta 0:00:06
   ----

## 2. PDF Upload & Folder Structure Setup
Upload source PDF guidelines (`NICE Asthma.pdf`, `WHO asthma.pdf`) or place them in `Docs/Sources/`.

In [2]:
os.makedirs("Docs/Sources", exist_ok=True)
os.makedirs("RAG/parsed_data", exist_ok=True)
os.makedirs("RAG/chunks_data", exist_ok=True)

try:
    from google.colab import files
    print("Please upload your medical PDF guidelines (e.g. NICE Asthma.pdf, WHO asthma.pdf) below:")
    uploaded = files.upload()
    for fname in uploaded.keys():
        os.rename(fname, os.path.join("Docs/Sources", fname))
    print("PDF files moved to Docs/Sources/")
except ImportError:
    print("Running in local environment. Ensure PDF files are placed in 'Docs/Sources/'")

Running in local environment. Ensure PDF files are placed in 'Docs/Sources/'


## 3. PDF Reader & Text Cleaning Module (`PDFParser`)
Parses raw PDF streams page-by-page while preserving 1-to-1 page citations, scrubbing recurring headers/footers (>=35% frequency), fixing hyphenated line splits, and stripping standalone page numbers.

In [3]:
import pypdf

class PDFParser:
    def __init__(self, remove_headers_footers: bool = True):
        self.remove_headers_footers = remove_headers_footers

    def extract_pages(self, pdf_path: str) -> List[Dict[str, Any]]:
        reader = pypdf.PdfReader(pdf_path)
        pages = []
        for idx, page in enumerate(reader.pages):
            text = page.extract_text() or ""
            pages.append({"page_number": idx + 1, "raw_text": text})
        return pages

    def detect_recurring_headers_footers(self, raw_pages: List[Dict[str, Any]]) -> Tuple[set, set]:
        top_counts, bottom_counts = {}, {}
        total = len(raw_pages)
        if total <= 2:
            return set(), set()
        for p in raw_pages:
            lines = [l.strip() for l in p["raw_text"].split("\n") if l.strip()]
            if lines:
                for l in lines[:2]:
                    if len(l) > 3 and not re.match(r"^\d+$", l):
                        top_counts[l] = top_counts.get(l, 0) + 1
                for l in lines[-2:]:
                    if len(l) > 3 and not re.match(r"^\d+$", l):
                        bottom_counts[l] = bottom_counts.get(l, 0) + 1
        thresh = max(3, int(total * 0.35))
        return {l for l, c in top_counts.items() if c >= thresh}, {l for l, c in bottom_counts.items() if c >= thresh}

    def clean_text(self, text: str, headers: set, footers: set) -> str:
        lines = text.split("\n")
        cleaned = []
        for l in lines:
            s = l.strip()
            if not s or s in headers or s in footers:
                continue
            if re.match(r"^(Page\s+\d+(\s+of\s+\d+)?|\d+\s*/\s*\d+|\d+)$", s, re.IGNORECASE):
                continue
            cleaned.append(l)
        res = "\n".join(cleaned)
        res = re.sub(r"(\b\w+)-\n(\w+\b)", r"\1\2", res)
        res = re.sub(r"[ \t]+", " ", res)
        return re.sub(r"\n{3,}", "\n\n", res).strip()

    def parse_document(self, pdf_path: str) -> Dict[str, Any]:
        fname = os.path.basename(pdf_path)
        dname = os.path.splitext(fname)[0]
        pages = self.extract_pages(pdf_path)
        headers, footers = self.detect_recurring_headers_footers(pages)
        parsed_pages = []
        for p in pages:
            ctext = self.clean_text(p["raw_text"], headers, footers)
            parsed_pages.append({"page_number": p["page_number"], "cleaned_text": ctext})
        return {"document_name": dname, "file_path": pdf_path, "pages": parsed_pages}

print("PDFParser ready!")

PDFParser ready!


## 4. Section-Aware Chunker (800 Tokens Size, 100 Tokens Overlap)
Splits text into 800-token chunks with 100-token overlap while respecting section headings and attaching metadata (`start_page`, `end_page`, `section_title`).

In [4]:
try:
    import tiktoken
    tokenizer = tiktoken.get_encoding("cl100k_base")
    def count_tokens(text: str) -> int:
        return len(tokenizer.encode(text))
except Exception:
    def count_tokens(text: str) -> int:
        return int(len(text.split()) * 1.3)

class SectionAwareChunker:
    def __init__(self, chunk_size: int = 800, chunk_overlap: int = 100):
        self.chunk_size = chunk_size
        self.chunk_overlap = chunk_overlap

    def _is_header(self, line: str) -> bool:
        s = line.strip()
        if not s:
            return False
        if re.match(r"^(\d+(\.\d+)*|Section\s+\d+|Part\s+\d+)\s+[:\-A-Z]", s, re.IGNORECASE):
            return True
        if s.isupper() and 4 <= len(s) <= 60 and not s.endswith("."): 
            return True
        return False

    def chunk_document(self, parsed_doc: Dict[str, Any]) -> List[Dict[str, Any]]:
        dname = parsed_doc["document_name"]
        fpath = parsed_doc["file_path"]
        chunks = []
        c_idx = 1
        curr_sec = "General / Overview"
        units = []
        curr_tokens = 0

        for p in parsed_doc["pages"]:
            pnum = p["page_number"]
            for line in p["cleaned_text"].split("\n"):
                lstr = line.strip()
                if not lstr:
                    continue
                ltokens = count_tokens(lstr)
                if self._is_header(lstr) and curr_tokens >= 200:
                    ctext = "\n".join([u["text"] for u in units]).strip()
                    chunks.append({
                        "chunk_id": f"{dname}_chunk_{c_idx:04d}",
                        "document_name": dname,
                        "file_path": fpath,
                        "section_title": curr_sec,
                        "start_page": units[0]["page"],
                        "end_page": units[-1]["page"],
                        "page_number": units[0]["page"],
                        "token_count": curr_tokens,
                        "text": ctext
                    })
                    c_idx += 1
                    ounits, otokens = [], 0
                    for u in reversed(units):
                        if otokens + u["tokens"] <= self.chunk_overlap:
                            ounits.insert(0, u)
                            otokens += u["tokens"]
                        else:
                            break
                    units, curr_tokens = ounits, otokens
                    curr_sec = lstr

                units.append({"text": lstr, "tokens": ltokens, "page": pnum})
                curr_tokens += ltokens

                if curr_tokens >= self.chunk_size:
                    ctext = "\n".join([u["text"] for u in units]).strip()
                    chunks.append({
                        "chunk_id": f"{dname}_chunk_{c_idx:04d}",
                        "document_name": dname,
                        "file_path": fpath,
                        "section_title": curr_sec,
                        "start_page": units[0]["page"],
                        "end_page": units[-1]["page"],
                        "page_number": units[0]["page"],
                        "token_count": curr_tokens,
                        "text": ctext
                    })
                    c_idx += 1
                    ounits, otokens = [], 0
                    for u in reversed(units):
                        if otokens + u["tokens"] <= self.chunk_overlap:
                            ounits.insert(0, u)
                            otokens += u["tokens"]
                        else:
                            break
                    units, curr_tokens = ounits, otokens

        if units:
            ctext = "\n".join([u["text"] for u in units]).strip()
            if len(ctext) > 20:
                chunks.append({
                    "chunk_id": f"{dname}_chunk_{c_idx:04d}",
                    "document_name": dname,
                    "file_path": fpath,
                    "section_title": curr_sec,
                    "start_page": units[0]["page"],
                    "end_page": units[-1]["page"],
                    "page_number": units[0]["page"],
                    "token_count": curr_tokens,
                    "text": ctext
                })
        return chunks

print("SectionAwareChunker ready!")

SectionAwareChunker ready!


## 5. Execute Ingestion Pipeline on Guidelines

In [5]:
parser = PDFParser()
chunker = SectionAwareChunker(chunk_size=800, chunk_overlap=100)

sources_dir = "Docs/Sources"
pdf_files = [f for f in os.listdir(sources_dir) if f.endswith(".pdf")] if os.path.exists(sources_dir) else []

all_chunks = []
for pf in pdf_files:
    ppath = os.path.join(sources_dir, pf)
    print(f"\n--- Processing: {pf} ---")
    doc_data = parser.parse_document(ppath)
    chunks = chunker.chunk_document(doc_data)
    print(f"Total Pages: {len(doc_data['pages'])} | Generated Chunks (800 size, 100 overlap): {len(chunks)}")
    all_chunks.extend(chunks)

print(f"\nPipeline Complete! Total Chunks Generated: {len(all_chunks)}")


--- Processing: GINA Asthma WMS.pdf ---


DependencyError: cryptography>=3.1 is required for AES algorithm

## 6. Clinical Retrieval Demonstration & Verification
Test clinical query retrieval over indexed chunks with document name, page number, and section citations.

In [ ]:
class SearchEngine:
    def __init__(self, chunks):
        self.chunks = chunks
        self.vocab = {}
        self.doc_vectors = []
        self._build()

    def _tok(self, text):
        return re.findall(r"\b\w+\b", text.lower())

    def _build(self):
        words = set()
        doc_toks = []
        for c in self.chunks:
            t = self._tok(c["text"])
            doc_toks.append(t)
            words.update(t)
        self.vocab = {w: i for i, w in enumerate(sorted(words))}
        for t in doc_toks:
            vec = np.zeros(len(self.vocab), dtype=np.float32)
            for w in t:
                if w in self.vocab:
                    vec[self.vocab[w]] += 1.0
            n = np.linalg.norm(vec)
            if n > 0:
                vec /= n
            self.doc_vectors.append(vec)
        self.doc_vectors = np.array(self.doc_vectors)

    def search(self, query, top_k=2):
        q_toks = self._tok(query)
        q_vec = np.zeros(len(self.vocab), dtype=np.float32)
        for w in q_toks:
            if w in self.vocab:
                q_vec[self.vocab[w]] += 1.0
        n = np.linalg.norm(q_vec)
        if n > 0:
            q_vec /= n
        scores = np.dot(self.doc_vectors, q_vec)
        top_idx = np.argsort(scores)[::-1][:top_k]
        return [{"score": float(scores[i]), "chunk": self.chunks[i]} for i in top_idx]

if all_chunks:
    engine = SearchEngine(all_chunks)
    queries = [
        "What is the recommended initial treatment for asthma in adults?",
        "How is asthma diagnosed in children according to WHO guidelines?",
        "When should inhaled corticosteroids (ICS) be prescribed?"
    ]
    for q in queries:
        print(f"\nQUERY: '{q}'")
        print("-" * 85)
        for res in engine.search(q, top_k=2):
            c = res["chunk"]
            print(f"  [Score: {res['score']:.4f}] Doc: {c['document_name']} | Page: {c['page_number']} | Section: {c['section_title']}")
            print(f"  Excerpt: {c['text'][:250]}...\n")